In [ ]:
# @title QR Code Generator { display-mode: "form" }

# --- 1. Install Libraries ---
%%capture
!pip install qrcode[pil] Pillow

# --- 2. Imports ---
import qrcode
from PIL import Image, ImageDraw, ImageFont
import os
from IPython.display import display

# Check for Colab environment
try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# ==========================================
# --- USER CONFIGURATION ---
# ==========================================

URL_DATA = "https://efrem.io/pcamp-2025" # @param {type:"string"}
BRAND_TEXT = "efremfilho" # @param {type:"string"}

# Text Size (Percentage of total image height)
TEXT_SIZE_PERCENTAGE = 7 # @param {type:"slider", min:5, max:50, step:1}

# Colors
FILL_COLOR = "black" # @param {type:"string"}
BACK_COLOR = "white" # @param {type:"string"}

# Resolution (30 is High Res)
BOX_SIZE = 30 # @param {type:"integer", min:10, max:50}

# QR Code Border Size (Standard is 4 modules)
# We need this constant to calculate exact alignment
QR_BORDER = 4

# ==========================================
# --- CORE LOGIC ---
# ==========================================

def get_font(size):
    """
    Tries to load a bold font installed in Colab/Linux.
    Falls back to a generic method if the specific file isn't found.
    """
    # Common path for fonts in Colab (Ubuntu)
    font_paths = [
        "/usr/share/fonts/truetype/liberation/LiberationSans-Bold.ttf",
        "/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf"
    ]

    selected_font_path = None
    for path in font_paths:
        if os.path.exists(path):
            selected_font_path = path
            break

    try:
        if selected_font_path:
            return ImageFont.truetype(selected_font_path, size)
        else:
            return ImageFont.truetype("arial.ttf", size)
    except OSError:
        print("WARNING: Could not load any TrueType font. Text will be tiny.")
        return ImageFont.load_default()

def generate_final_qr(data, text, fill, back, box_size, text_pct):
    print("Generating QR Code...")

    # 1. Create QR
    qr = qrcode.QRCode(
        error_correction=qrcode.constants.ERROR_CORRECT_H,
        box_size=box_size,
        border=QR_BORDER, # Using the constant defined above
    )
    qr.add_data(data)
    qr.make(fit=True)

    img_qr = qr.make_image(fill_color=fill, back_color=back).convert('RGB')

    if not text:
        return img_qr

    # 2. Setup Drawing
    draw = ImageDraw.Draw(img_qr)
    width, height = img_qr.size

    # 3. Calculate Font Size
    target_font_size = int(height * (text_pct / 100))
    font = get_font(target_font_size)

    # 4. Measure Text
    left, top, right, bottom = draw.textbbox((0, 0), text, font=font)
    text_w = right - left
    text_h = bottom - top

    # 5. Position Calculation (ALIGNED TO QR MODULES)
    # Calculate the exact width of the white border in pixels
    quiet_zone_pixels = QR_BORDER * box_size

    # X: Image Width - Quiet Zone - Text Width
    # This aligns the end of the text with the last black column of the QR code
    x = width - quiet_zone_pixels - text_w

    # Y: Image Height - Quiet Zone - Text Height
    # This aligns the bottom of the text with the last black row of the QR code
    y = height - quiet_zone_pixels

    # 6. Draw Background Box for Text
    # We add a small margin to the box so the text "breathes"
    box_padding = int(target_font_size * 0.2)
    draw.rectangle(
        [
            x - box_padding,
            y - box_padding,
            x + text_w + box_padding,
            y + text_h + box_padding
        ],
        fill=back
    )

    # 7. Draw Text
    # We adjust the Y slightly because text fonts have internal padding/ascenders
    draw.text((x, y - (top * 0.1)), text, font=font, fill=fill)

    print(f"Done! Resolution: {width}x{height}px.")
    return img_qr

# ==========================================
# --- EXECUTION ---
# ==========================================

final_image = generate_final_qr(
    URL_DATA,
    BRAND_TEXT,
    FILL_COLOR,
    BACK_COLOR,
    BOX_SIZE,
    TEXT_SIZE_PERCENTAGE
)

display(final_image)

if IN_COLAB:
    save_name = "qr_code_aligned.png"
    final_image.save(save_name)
    try:
        files.download(save_name)
    except Exception:
        print(f"Download failed. File saved as {save_name}")